# AST Index Notebook

Provides a simple workflow for parsing code into a Tree-sitter AST and building structural metadata.

In [ ]:
from __future__ import annotations
from typing import Any, Dict, List, Optional
class ASTIndexer:
    """Parses source code into a Tree-sitter AST and extracts lightweight
    structural metadata (function/class names, node count) for each snippet.
    Falls back to a naive line/keyword count if tree-sitter isn't available,
    so the rest of the pipeline can still run without it installed."""

    def __init__(self, language: str = "python"):
        self.language = language
        self._parser = None
        self._ready = False
        self._init_parser()

    def _init_parser(self) -> None:
        if self.language != "python":
            return  # only python grammar is wired up by default
        try:
            import tree_sitter_python as tspython
            from tree_sitter import Language, Parser

            py_language = Language(tspython.language())
            parser = Parser(py_language)
            self._parser = parser
            self._ready = True
        except ImportError:
            self._ready = False

    def index_code(self, code: str) -> Dict[str, Any]:
        if self._ready:
            return self._index_with_tree_sitter(code)
        return self._index_naive(code)

    def _index_with_tree_sitter(self, code: str) -> Dict[str, Any]:
        tree = self._parser.parse(bytes(code, "utf-8"))
        root = tree.root_node

        functions, classes = [], []

        def walk(node):
            if node.type == "function_definition":
                name_node = node.child_by_field_name("name")
                if name_node is not None:
                    functions.append(code[name_node.start_byte:name_node.end_byte])
            elif node.type == "class_definition":
                name_node = node.child_by_field_name("name")
                if name_node is not None:
                    classes.append(code[name_node.start_byte:name_node.end_byte])
            for child in node.children:
                walk(child)

        walk(root)

        return {
            "code": code,
            "language": self.language,
            "node_count": root.descendant_count if hasattr(root, "descendant_count") else None,
            "functions": functions,
            "classes": classes,
            "has_error": root.has_error,
        }

    @staticmethod
    def _index_naive(code: str) -> Dict[str, Any]:
        functions = [
            line.split("def ", 1)[1].split("(")[0].strip()
            for line in code.splitlines() if line.strip().startswith("def ")
        ]
        classes = [
            line.split("class ", 1)[1].split("(")[0].split(":")[0].strip()
            for line in code.splitlines() if line.strip().startswith("class ")
        ]
        return {
            "code": code, "language": "python", "node_count": None,
            "functions": functions, "classes": classes, "has_error": None,
        }

In [ ]:
class ASTIndexManager:
    """Holds a corpus of AST records built via ASTIndexer, for structural
    lookup/fusion alongside the semantic (FAISS) index."""

    def __init__(self, language: str = "python"):
        self.indexer = ASTIndexer(language=language)
        self.records: List[Dict[str, Any]] = []

    def build_ast_index(self, code_samples: List[str]) -> List[Dict[str, Any]]:
        self.records = [self.indexer.index_code(code) for code in code_samples]
        return self.records

    def find_by_function_name(self, name: str) -> List[Dict[str, Any]]:
        return [r for r in self.records if name in r.get("functions", [])]

In [ ]:
TreeSitterASTIndexer = ASTIndexManager